<a href="https://colab.research.google.com/github/suhaniagarwal18/ecommerce-customer-segmentation-rfm/blob/main/notebooks/rfm_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E-commerce Customer Segmentation using RFM Analysis

This project segments the customers of an online retailer using RFM analysis (Recency, Frequency, Monetary value). The goal is to identify which customers are most valuable, which are at risk of leaving, and what action to take for each group.

**Tools:** Python, pandas, matplotlib, seaborn (Google Colab)

## 1. Business problem

Most businesses treat all customers the same, which wastes marketing budget on people who would buy anyway and ignores those about to leave. RFM analysis scores every customer on three questions:

- **Recency:** how recently did they buy?
- **Frequency:** how often do they buy?
- **Monetary:** how much do they spend?

Grouping customers by these scores lets the business target each group with the right strategy, such as rewarding loyal customers or winning back inactive ones.

## 2. Data understanding

The data comes from the [Online Retail Dataset on Kaggle](https://www.kaggle.com/datasets/divanshu22/online-retail-dataset). It contains transactions from an online retailer. Each row is one product line on an invoice, so a single purchase (invoice) can span several rows.

| Column | Meaning | Used for |
|---|---|---|
| InvoiceNo | Invoice number, one per purchase. A "C" prefix marks a cancellation | Frequency |
| StockCode | Product code | Not needed for RFM |
| Description | Product name | Not needed for RFM |
| Quantity | Units bought in that line | Monetary |
| InvoiceDate | Date and time of the purchase | Recency, Frequency |
| UnitPrice | Price per unit | Monetary |
| CustomerID | Unique customer identifier | Grouping key |
| Country | Customer's country | Optional context |

In [1]:
import pandas as pd
df = pd.read_excel("customer.xlsx")

This tells us about the no of rows and columns of the dataset

In [2]:
df.shape

(541909, 8)

The dataset has 541,909 rows and 8 columns.

In [3]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


The first rows show the layout: several rows share one InvoiceNo, confirming that each row is a product line within a purchase.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


InvoiceDate is already a datetime, which is what we need for Recency. CustomerID is stored as a float (not an integer) because it contains missing values. We will convert it after cleaning.

In [5]:
df.isnull().sum()

,0
InvoiceNo,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135080
Country,0


CustomerID is missing in 135,080 rows (about 24.9%), and Description in 1,454 rows. RFM needs to know who each customer is, so the missing CustomerID rows are the most important problem to solve.

In [6]:
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386048,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


- The data covers 1 December 2010 to 9 December 2011, about one year.
- Quantity has negative values (minimum -80,995).
- UnitPrice also has a negative value (minimum -11,062.06), which cannot be a real sale.

In [7]:
df[df['Quantity']<0].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,2010-12-01 10:24:00,3.45,17548.0,United Kingdom
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
939,C536506,22960,JAM MAKING SET WITH JARS,-6,2010-12-01 12:38:00,4.25,17897.0,United Kingdom


In [8]:
df[df['UnitPrice']<=0].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom


### Issues found

| Issue | Evidence | Planned action |
|---|---|---|
| Missing CustomerID | 135,080 rows (24.9%) | [fill in after deciding] |
| Cancelled orders | InvoiceNo starts with "C" and Quantity is negative | [fill in after deciding] |
| Non-product rows | "Discount" and "Adjust bad debt" lines, the latter with a negative UnitPrice and no CustomerID | [fill in after deciding] |
| Duplicate rows | 5,268 identical rows | Remove |

## 3. Data cleaning

Each cleaning step is applied separately, and the number of rows before and after is printed, so every decision can be traced.

In [9]:
df.duplicated().sum()

np.int64(5268)

### Step 1: Remove duplicate rows

Rows that are identical in every column are very likely repeated entries of the same record, so keeping them would count the same purchase twice.

In [10]:
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} rows. Rows Left: {len(df)}")

Removed 5268 rows. Rows Left: 536641


5,268 duplicate rows (about 1% of the data) were removed, leaving 536,641 rows. One limitation is that a customer could genuinely have bought the same item twice in the same moment, but at 1% the effect on the analysis is small.